# Métricas de Regressão e Regularização

Avaliação e refinamento de modelos de regressão.

**Conceitos:**
- Métricas: MAE, MSE, RMSE, MAPE, R²
- Overfitting em polinômios de alto grau
- Regularização L2 (Ridge) e L1 (Lasso)

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error,
                             mean_squared_error,
                             mean_absolute_percentage_error,
                             r2_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso

## 2. Geração dos Dados

In [ ]:
# Define a semente do gerador de números aleatórios para garantir reprodutibilidade
np.random.seed(42)
# Gera 100 amostras para a variável independente X com distribuição normal (média 0, desvio 2)
X = 2 * np.random.randn(100, 1)
# Gera a variável dependente y a partir de uma relação linear: y = 4 + 3*X + ruído gaussiano
y = 4 + 3 * X + np.random.randn(100, 1)

# Cria um gráfico de dispersão para visualizar a relação entre X e y
plt.scatter(X, y, c='blue')
plt.title('Dados sintéticos lineares')
plt.xlabel('X')
plt.ylabel('y')
plt.grid()
plt.show()

## 3. Treinamento do Modelo

In [ ]:
# Cria uma instância do modelo de regressão linear
lin_reg = LinearRegression()
# Treina o modelo com os dados X e y, ajustando os coeficientes por mínimos quadrados
lin_reg.fit(X, y)
# Exibe o intercepto (w0) e o coeficiente angular (w1) aprendidos pelo modelo
print(f'W0: {lin_reg.intercept_}')
print(f'W1: {lin_reg.coef_}')

## 4. Métricas de Erro

In [ ]:
# Gera as previsões do modelo para os mesmos dados de treino
y_pred = lin_reg.predict(X)

# Calcula o Erro Médio Absoluto (MAE) — média dos valores absolutos dos resíduos
ema = mean_absolute_error(y, y_pred)
# Calcula o Erro Quadrático Médio (MSE) — penaliza erros grandes por elevá-los ao quadrado
eqm = mean_squared_error(y, y_pred)
# Calcula a Raiz do Erro Quadrático Médio (RMSE) — mesma unidade da variável alvo
rqm = np.sqrt(eqm)
# Calcula o Erro Percentual Absoluto Médio (MAPE) — erro relativo em porcentagem
epam = mean_absolute_percentage_error(y, y_pred)
# Calcula o Coeficiente de Determinação (R²) — proporção da variância explicada pelo modelo
r2 = r2_score(y, y_pred)

# Exibe todas as métricas calculadas de forma organizada
print(f"""
EMA: {ema:.2f}
EQM: {eqm:.2f}
RQM: {rqm:.2f}
EPAM: {epam:.2f}
R² {r2:.2f}
""")

## 5. Overfitting

In [ ]:
# Define a semente para garantir que os resultados sejam reproduzíveis
np.random.seed(42)
m = 20  # Usa poucos dados para que o modelo complexo possa se ajustar excessivamente
# Gera dados não-lineares que seguem uma curva senoidal com adição de ruído gaussiano
X = 3 * np.random.rand(m, 1)
y = np.sin(X).ravel() + np.random.randn(m) * 0.2

# Cria uma grade uniforme de pontos no intervalo [0, 3] para plotar a curva suave do modelo
X_plot = np.linspace(0, 3, 100).reshape(-1, 1)

# Cria um modelo polinomial de grau 15 — muito complexo para apenas 20 amostras
grau = 15
modelo_over = make_pipeline(
    PolynomialFeatures(grau),
    LinearRegression()
)
# Treina o modelo polinomial sobre os dados ruidosos
modelo_over.fit(X, y)

# Faz previsões ao longo de toda a grade para visualizar a curva ajustada
y_plot_over = modelo_over.predict(X_plot)

# Plota os dados reais, a curva senoidal verdadeira e a curva do modelo overfitado
plt.scatter(X, y, color='red', label='Dados Treino')
plt.plot(X_plot, np.sin(X_plot), 'g--', label='Curva Real (Seno)')
plt.plot(X_plot, y_plot_over, color='blue', label='Modelo Overfitting')
plt.ylim(-2, 2)
plt.legend()
plt.grid()
plt.title('O Perigo do Overfitting (grau 15)')
plt.show()

## 6. Regularização Ridge e Lasso

In [ ]:
# --- RIDGE (Regularização L2) ---
# Alpha controla a intensidade da regularização: quanto maior alpha, mais o modelo é simplificado
# Ridge adiciona uma penalidade proporcional ao quadrado dos coeficientes
reg_ridge = make_pipeline(
    PolynomialFeatures(grau),
    Ridge(alpha=1)
)
reg_ridge.fit(X, y)
y_plot_ridge = reg_ridge.predict(X_plot)

# --- LASSO (Regularização L1) ---
# Lasso adiciona uma penalidade proporcional ao valor absoluto dos coeficientes, podendo zerá-los
# Requer mais iterações para convergir devido à natureza da otimização
reg_lasso = make_pipeline(
    PolynomialFeatures(grau),
    Lasso(alpha=0.01, max_iter=10_000)
)
reg_lasso.fit(X, y)
y_plot_lasso = reg_lasso.predict(X_plot)

# Gráfico comparativo entre os modelos sem regularização, com Ridge e com Lasso
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='red', label='Dados Treino')
plt.plot(X_plot, np.sin(X_plot), 'g--', label='Real (Seno)', alpha=0.5)

# Curva do modelo polinomial sem regularização (overfitting evidente)
plt.plot(X_plot, y_plot_over, 'b:', label='Sem regularização (overfit)', alpha=0.3)

# Curvas dos modelos regularizados — mais suaves e próximas da curva real
plt.plot(X_plot, y_plot_ridge, 'orange', linewidth=2, label='Ridge (L2)')
plt.plot(X_plot, y_plot_lasso, 'purple', linewidth=2, label='Lasso (L1)')

plt.ylim(-2, 2)
plt.legend()
plt.grid()
plt.title('Regularização salvando o modelo')
plt.show()